<a href="https://colab.research.google.com/github/bhanukiran29/AILAB_1WN24CS150/blob/main/1WN24CS150_Week6_PropositionalLogic_TruthTableEnumeration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Propositional Entailment via Truth-Table Enumeration (`TT-ENTAILS`)

Following the provided textbook algorithm, we implement:
1. `PL_TRUE(sentence, model)`: Determines if a sentence is true in a given partial (or full) model.
2. `TT_ENTAILS(KB, alpha)`: Returns `True` if the Knowledge Base $KB$ entails the query $\alpha$, and `False` otherwise.
3. `TT_CHECK_ALL(KB, alpha, symbols, model)`: Recursively checks all truth assignments for the symbols.

In [3]:
def PL_TRUE(sentence, model):
    """
    Returns True if the sentence holds within the model (assignment of truth values).
    We use Python's eval function mapping variables to their values in the model.
    """
    try:
        # Convert standard logical words to python operators if needed
        processed_sentence = sentence.replace("~", " not ").replace("&", " and ").replace("|", " or ")
        return bool(eval(processed_sentence, {"__builtins__": None}, model))
    except Exception as e:
        # In case the model is partial and doesn't contain all variables yet
        raise ValueError(f"Error evaluating '{sentence}' with model {model}: {e}")

def TT_ENTAILS(KB, alpha, symbols=None):
    """
    Decides if the knowledge base KB entails the query alpha.
    """
    if symbols is None:
        # Automatically extract proposition symbols (simple alphanumeric words)
        import re
        all_words = re.findall(r'\b[a-zA-Z_][a-zA-Z0-9_]*\b', f"{KB} {alpha}")
        # Exclude logical keywords
        keywords = {'and', 'or', 'not', 'True', 'False'}
        symbols = list(set(w for w in all_words if w not in keywords))

    return TT_CHECK_ALL(KB, alpha, symbols, {})

def TT_CHECK_ALL(KB, alpha, symbols, model):
    """
    Recursively checks all truth assignments for the symbols.
    """
    if not symbols:
        if PL_TRUE(KB, model):
            return PL_TRUE(alpha, model)
        else:
            return True  # when KB is false, always return true
    else:
        P = symbols[0]
        rest = symbols[1:]

        # Create copies of model with P set to True and False
        model_true = model.copy()
        model_true[P] = True

        model_false = model.copy()
        model_false[P] = False

        return (
            TT_CHECK_ALL(KB, alpha, rest, model_true)
            and
            TT_CHECK_ALL(KB, alpha, rest, model_false)
        )

### Verification / Example

Let's test the implementation with a simple logical deduction:
* **Knowledge Base (KB)**: `(A or B) and (not A)` (Either A or B is true, and A is not true).
* **Query ($\alpha$)**: `B` (Therefore, B must be true).

If $KB \models \alpha$, `TT_ENTAILS` should return `True`.

In [4]:
KB = "(A or B) and (not A)"
alpha = "B"

result = TT_ENTAILS(KB, alpha)
print(f"Does KB entail alpha? {result}")

Does KB entail alpha? True


### Truth Table for the Entailment Example

Let's visualize the truth table for the same variables `A` and `B` with the columns showing:
1. The values of `A` and `B`.
2. The status of the Knowledge Base ($KB$): `(A or B) and (not A)`.
3. The status of the Query ($\alpha$): `B`.
4. Whether the entailment holds for that row (implied as $KB \implies \alpha$).

In [5]:
import itertools

# Variables involved
vars_list = ['A', 'B']
combinations = list(itertools.product([True, False], repeat=2))

print(f"{'A':<6} | {'B':<6} | {'KB: (A or B) and (not A)':<30} | {'Query: B':<10} | {'KB => Query':<12}")
print("-" * 73)

for A, B in combinations:
    model = {'A': A, 'B': B}
    kb_val = PL_TRUE("(A or B) and (not A)", model)
    query_val = PL_TRUE("B", model)
    # Implication KB => Query is True if KB is False or Query is True
    entails_val = (not kb_val) or query_val

    print(f"{str(A):<6} | {str(B):<6} | {str(kb_val):<30} | {str(query_val):<10} | {str(entails_val):<12}")

A      | B      | KB: (A or B) and (not A)       | Query: B   | KB => Query 
-------------------------------------------------------------------------
True   | True   | False                          | True       | True        
True   | False  | False                          | False      | True        
False  | True   | True                           | True       | True        
False  | False  | False                          | False      | True        


### Simpler KB Example

Let's test with a much simpler logic setting:
* **KB**: `A` (We know $A$ is true).
* **Query ($\alpha$)**: `A or B` (We want to verify if $A \implies A \lor B$).

In [6]:
simple_KB = "A"
simple_alpha = "A or B"

simple_result = TT_ENTAILS(simple_KB, simple_alpha)
print(f"Does simple KB '{simple_KB}' entail '{simple_alpha}'? {simple_result}")

Does simple KB 'A' entail 'A or B'? True


### Truth Table for the Simpler KB

In [7]:
print(f"{'A':<6} | {'B':<6} | {'KB: A':<10} | {'Query: A or B':<15} | {'KB => Query':<12}")
print("-" * 59)

for A, B in combinations:
    model = {'A': A, 'B': B}
    kb_val = PL_TRUE(simple_KB, model)
    query_val = PL_TRUE(simple_alpha, model)
    entails_val = (not kb_val) or query_val

    print(f"{str(A):<6} | {str(B):<6} | {str(kb_val):<10} | {str(query_val):<15} | {str(entails_val):<12}")

A      | B      | KB: A      | Query: A or B   | KB => Query 
-----------------------------------------------------------
True   | True   | True       | True            | True        
True   | False  | True       | True            | True        
False  | True   | False      | True            | True        
False  | False  | False      | False           | True        
